#### Case Study 2: Representing Clinical Text as Vectors

Two ways to represent text as vectors are TF-IDF and word embeddings. Here, I will walk through implementing TF-IDF on MIMIC-III discharge summaries and begin exploring simple embeddings from PubMed. 

The data is a subset of the MIMIC-III discharge summaries. This requires some credentials before data access.

LLM Statement: I approach each calculation and observation myself and used LLM to verify my logic and claims.

In [1]:
import pandas as pd
import nltk

from math import log
from collections import defaultdict
import random

In [2]:
discharge_summaries = pd.read_csv('data.csv')
print(discharge_summaries)

       HADM_ID                                               TEXT
0     121936.0  Admission Date:  [**2125-2-9**]              D...
1     155131.0  Admission Date:  [**2131-12-23**]             ...
2     113012.0  Admission Date:  [**2194-1-7**]              D...
3     125055.0  Admission Date:  [**2189-3-17**]       Dischar...
4     139542.0  Admission Date:  [**2146-7-1**]              D...
...        ...                                                ...
3663  132211.0  Name:  [**Known lastname 17303**], [**Known fi...
3664  190484.0  Name:  [**Known lastname 400**],[**Known first...
3665  109285.0  Name:  [**Known lastname 18278**],[**Known fir...
3666  183977.0  Name:  [**Known lastname 18438**],[**Known fir...
3667  155984.0  Name:  [**Known lastname **],[**Known firstnam...

[3668 rows x 2 columns]


From the 3668 discharge summaries from the MIMIC-III dataset, I'll first define the corpus and convert to lowercase. Text normalization like this is important so that words like "Sepsis" and "sepsis" can be counted as one.

In [3]:
corpus = discharge_summaries.TEXT.str.lower()
print(corpus[0])

admission date:  [**2125-2-9**]              discharge date:   [**2125-2-16**]


service: medicine

allergies:
zocor / lescol

attending:[**doctor last name 1857**]
chief complaint:
chest pain

major surgical or invasive procedure:
central venous line insertion (right internal jugular vein)

history of present illness:
mr. [**known lastname 1858**] is an 84 yo man with moderate aortic stenosis (outside
hospital echo in [**2124**] with [**location (un) 109**] 1 cm2, gradient 28 mmhg, moderate
mitral regurgitation, mild aortic insufficiency), chronic left
ventricular systolic heart failure with ef 25-30%, hypertension,
hyperlipidemia, diabetes mellitus, cad s/p cabg in [**2099**] with
svg-lad-diagonal, svg-om, and svg-rpda-rpl, with a re-do cabg in
[**9-/2117**] with lima-lad, svg-om, svg-diagonal, and svg-rca. he also
has severe peripheral arterial disease s/p peripheral bypass
surgery. he presented to [**hospital 1474**] hospital er this morning with
shortness of breath and chest pain 

#### TF-IDF
To implement TF-IDF, I will need to define a $1 \times N$ vocabulary vector. Let's take a look at the distribution of terms across our corpus.

In [4]:
def count_words(corpus: list) -> list:
    word_counts = defaultdict(int)
    for document in corpus:
        words = nltk.word_tokenize(document)
        for word in words:
            word_counts[word] += 1

    return word_counts

word_counts = count_words(corpus)

In [5]:
# do this for speed
tokenized_corpus = [nltk.word_tokenize(document) for document in corpus]

#### Gathering useful findings about the words in our corpus
- 122605 unique words are in our corpus
- 61248 unique words occur exactly once in our corpus

There are going to some words we rarely use. To capture words that are actually used:
- 27668 nique words in our corpus have a count > 5

In [6]:
len(word_counts)

122605

In [7]:
one_unique_words = 0
for word, count in word_counts.items():
    if count  == 1:
        one_unique_words += 1
print(one_unique_words)

61248


In [8]:
greater_than_five_unique_words = 0
for word, count in word_counts.items():
    if count  > 5:
        greater_than_five_unique_words += 1
print(greater_than_five_unique_words)

27668


As seen above, many words occur only once in our corpus. As discussed in class, when creating a vector of word counts this results in a very sparse representation. We can mitigate this somewhat with preprocessing, lemmatization, better tokenization, and by only preserving common words. Here, let's define our vocabulary as any word with a count > 5.

In [9]:
MIN_COUNT = 5
def create_vocabulary(corpus: list) -> list:
    vocabulary = []
    word_counts = count_words(corpus)
    for word, count in word_counts.items():
        if count > MIN_COUNT:
            vocabulary.append(word)
    return vocabulary

vocabulary = create_vocabulary(corpus)

#### Create a TF vector for each document

Since the term frequency is the number of times the term appeared in one document, and recall, there are $3,668$ documents.

In [10]:
# defaultdic for speed
def calculate_tf(corpus: list, vocabulary: list) -> list:
    tfs = []
    for words in corpus:
        # get a document level vector since it'll vary per document
        doc_vec = []
        # helps you count faster
        counter = defaultdict(int)
        for w in words:
            # count the number of times a word appear in the document
            counter[w] += 1
        for vocab in vocabulary:
            doc_vec.append(counter.get(vocab, 0))
        tfs.append(doc_vec)
    return tfs
        
tfs = calculate_tf(tokenized_corpus, vocabulary)

In [11]:
print(len(tfs), len(tfs[0]))

3668 27668


#### Create the IDF vector

$$\text{idf}(t) = \log\left(\frac{N}{\text{df}(t)}\right)$$

In [12]:
len(corpus) 

3668

In [13]:
len(tokenized_corpus)

3668

In [14]:
def calculate_idf(corpus: list, vocabulary: list) -> list:
    # count how many documents the word appears in 
    # n is the length of the corpus
    n = len(corpus)
    idf = []
    # speed: go through the corpus once and count every word as you go
    doc_freqs = defaultdict(int)
    for document in corpus:
        for word in set(document):
            doc_freqs[word] += 1
    idf = []
    for word in vocabulary:
        idf.append(log(n / doc_freqs[word]))
    return idf

idf = calculate_idf(tokenized_corpus, vocabulary)

#### Combine our TF and IDF implementations to calculate the TF-IDF for each discharge summary

In [15]:
print("idf columns:", len(idf))
print("tfs rows:", len(tfs[0]))

idf columns: 27668
tfs rows: 27668


In [16]:
def calculate_tf_idf(tf: list, idf: list) -> list:
    tf_idf = []
    for i in range(len(tf)):
        result = [term * inverse for term, inverse  in zip(tf[i], idf)]
        tf_idf.append(result)
    return tf_idf


tf_idfs = calculate_tf_idf(tfs, idf)

In [17]:
print(tf_idfs[:5]) # as expected, vocab has lots of words so it's sparse

[[0.004364430329338228, 0.0005453306100587508, 0.010906612201175017, 0.014451261166556897, 0.06489434259699135, 49.24441100025813, 0.014451261166556897, 0.031160381639446417, 0.004097806581073031, 0.4452312262381508, 0.09128340176698961, 3.1704492309627264, 0.9936335252577141, 5.262962854209915, 0.32282532278003107, 1.4550754044226586, 0.10147904022660706, 0.038536837731753974, 0.2807992341949712, 0.2739634457488665, 4.271683196768543, 2.9307539132594145, 0.3537970202785189, 0.31944249677641057, 0.8147503066846938, 0.363161115234544, 0.28622911178934063, 1.4206848827712746, 2.467717627712431, 1.0054855158447285, 3.497871632064021, 0.00654396732070501, 1.2366683649447006, 3.8169051730408254, 3.82058502688484, 1.8154847199837534, 0.009815950981057515, 0.5623875498964331, 0.010906612201175017, 0.16058529605996, 0.08236273600861001, 4.617520754876856, 0.8025907710434192, 2.6606246133108478, 1.4635802466804477, 0.6034195653622509, 5.355945491729864, 2.80335855991874, 3.6511716319405148, 0.9

Let's take a look at the TF-IDF representation of one of our discharge summaries

In [18]:
print(tf_idfs[random.randint(0, len(corpus))])

[0.007637753076341899, 0.0013633265251468771, 0.03353783251861318, 0.022903885622467535, 0.10443081182625077, 0.0, 0.022903885622467535, 0.04154717551926189, 0.008195613162146061, 0.4452312262381508, 0.09128340176698961, 0.0, 1.9872670505154282, 0.0, 0.32282532278003107, 2.1826131066339878, 0.27061077393761884, 0.1348789320611389, 0.2807992341949712, 0.2739634457488665, 0.7538264464885664, 1.1723015653037656, 0.3537970202785189, 0.31944249677641057, 1.2221254600270408, 0.363161115234544, 0.28622911178934063, 0.0, 0.0, 0.0, 0.0, 0.022358555012408785, 0.17666690927781437, 0.0, 0.0, 0.0, 0.022358555012408785, 0.7029844373705414, 0.008997955065969388, 0.24087794408994, 0.08236273600861001, 0.0, 0.0, 0.0, 0.42687757194846393, 0.12068391307245019, 0.0, 1.40167927995937, 0.0, 0.39092779361463204, 0.0, 0.0, 0.0, 0.0, 0.13294637337942095, 0.0, 0.4824159440743248, 0.0, 1.2728865004146903, 0.9685634822200478, 0.0, 1.6118311695312022, 0.0, 0.04907975490528757, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2774825014

We now have a $3668 \times 27668$ matrix representation of our corpus. One simple use for this is search: given one patient's discharge summary can we find a similar one? To do this we can use the cosine similarity between any two vectors in our corpus.

In [19]:
import numpy as np
def calculate_cos_similarity(idx, tf_idfs):
    vector = tf_idfs[idx]
    cos_similarities = []
    for i, v in enumerate(tf_idfs):
        if i != idx:
            cos = (
                np.dot(vector, v)
                /
                (np.linalg.norm(vector)*np.linalg.norm(v))
            )

            cos_similarities.append(cos)
        else:
            cos_similarities.append(0)

    return cos_similarities

cos_similarities = calculate_cos_similarity(1000, tf_idfs)

In [20]:
print(corpus[1000])

admission date:  [**2118-5-26**]              discharge date:   [**2118-5-28**]


service: neurology

allergies:
sulfa (sulfonamide antibiotics) / penicillins / penicillins /
clonidine / clonidine / azithromycin

attending:[**first name3 (lf) 5018**]
chief complaint:
r sided weakness

major surgical or invasive procedure:
tpa administration
right femoral line placement
intubation


history of present illness:
reason for consult; code stroke

time code stroke called: 0850 (24h clock)
time neurology at bedside for evaluation: 0855
time (and date) the patient was last known well: 0820
nih stroke scale score: 20
t-[**md number(3) 6360**]: yes
time t-pa was given: 0944

i was present during the ct scanning and reviewed the images
instantly within 20 minutes of their completion.


nih stroke scale score was 20:
1a. level of consciousness: 0
1b. loc question: 2
1c. loc commands: 2
2. best gaze: 2
3. visual fields: 2
4. facial palsy: 2
5a. motor arm, left: 0
5b. motor arm, right: 4
6a. motor l

In [21]:
print(corpus[np.argmax(cos_similarities)])

admission date:  [**2192-7-30**]              discharge date:   [**2192-8-3**]

date of birth:  [**2130-6-29**]             sex:   f

service: neurology

allergies:
no known allergies / adverse drug reactions

attending:[**last name (namepattern1) 1838**]
chief complaint:
left sided weakness, aphasia

major surgical or invasive procedure:
tpa

history of present illness:
62 yo f with hx htn and hld was at work this afternoon and
reported to not be speaking much around 2 pm onward.  at 4 pm
she developed global aphasia and right hemiplegia and ems was
called. no further information was available at time code stroke
was called.


past medical history:
- htn
- hld
- cad s/p cabg/avr [**12-13**]
- dilated cardiomyopathy
- systolic chf with ef 20%

social history:
pt reports that she used to live alone but since her recent
cardiac surgery her daughter has moved in with her.  resides in
[**location (un) 686**]. has a history of etoh abuse and cocaine use. has
been decreasing her alcohol cons

#### What similarities, if any, between these two documents?

There's a lot of overlap between these documents. First,they are ...

#### Word Embeddings
An approach to learning is word embeddings. Here's a pretrained embeddings from pubmed that I will use for this case study.

In [22]:
from gensim.models.keyedvectors import KeyedVectors
word_vectors_pubmed = KeyedVectors.load_word2vec_format('pubmed2018_w2v_200D.bin',binary=True, limit=200000)

Here are what the embeddings look like:

In [23]:
print("Word:", 'sepsis')
print("Embedding:", word_vectors_pubmed['sepsis'])

Word: sepsis
Embedding: [-0.43890506 -0.30916336 -0.01412137  0.3092074  -0.44178864 -0.11187912
 -0.21391366  0.13776776  0.32003063 -0.20084469 -0.27083948 -0.14592735
  0.22776209  0.14080325  0.13969716  0.08312299 -0.1541089   0.04092164
  0.08164905  0.09110633 -0.38671106 -0.38058448  0.13552871 -0.34116757
 -0.02990485  0.21664007  0.10085899  0.548211    0.2421064  -0.3445053
  0.72779274 -0.0950932  -0.1639559  -0.18408148 -0.34428382 -0.7541353
 -0.07928886 -0.16611007  0.08586685 -0.44759     0.12478107  0.02663528
 -0.26105708  0.35435095 -0.14034367  0.06383427  0.00835935 -0.07049082
 -0.2582347   0.10057217 -0.3829518  -0.36446834 -0.44020787 -0.29055005
 -0.00644601 -0.13146453 -0.3911157  -0.30744675 -0.10724983  0.32979518
 -0.11416943  0.03893044 -0.09785876  0.48078305  0.15236712  0.16935015
 -0.01975257  0.38183266 -0.03936606 -0.17248844  0.02380012 -0.4596762
 -0.2409243  -0.2849568  -0.346123    0.31556925  0.00786234  0.4428732
 -0.1585959   0.37303257  0.181

We can use the ```most_similar``` that implements the cosine similarity, with the ```positive``` keyword for vector addition and the ```negative``` keyword for vector subtraction.

In [24]:
word_vectors_pubmed.most_similar(positive=['severe', 'infection'])

[('infections', 0.7753844857215881),
 ('septicemia', 0.6676729917526245),
 ('co-infection', 0.6442705392837524),
 ('more-severe', 0.6407489776611328),
 ('serious', 0.64056396484375),
 ('septicaemia', 0.6367871761322021),
 ('infected', 0.6338090300559998),
 ('fulminating', 0.6272422075271606),
 ('pneumonia', 0.6266636252403259),
 ('coinfection', 0.6244802474975586)]

#### Observations

The words outputed by the pubmed2018 word2vec are related in medical meaning. This was something we did not see with tf-idf. Since word2vec captured the context of the words and grouped words that appear in similar contexts together, the embedding model was able to output more meaningful similar words.

#### Experiment 
Try some other vector arithmetic of interest.

In [25]:
word_vectors_pubmed.most_similar(positive=['lung', 'disease'])

[('diseases', 0.6660116910934448),
 ('dild', 0.6355810165405273),
 ('pulmonary', 0.6347777247428894),
 ('copd', 0.6269057989120483),
 ('lungs', 0.6060346364974976),
 ('diseasethe', 0.6047483086585999),
 ('ipf', 0.602691650390625),
 ('non-small', 0.5998053550720215),
 ('emphysema', 0.5989214181900024),
 ('kidney', 0.5967196226119995)]

As we can see, ```diseasethe```is most likely a broken token and ```kidney``` is a not quite right too, but other than that, the similarities are mostly reasonable and meaningful.